# AGHRI 2D tracking: paper tables and figures

This notebook reproduces the two 2D tracking tables and two figures used in the
manuscript. The evaluation covers all six held-out AGHRI test sequences in
ZED RGB and front fisheye (12 camera-specific sequences) and eight trackers.
Detector-based results use the validation-selected YOLOv8s checkpoint for each
camera; ground-truth boxes provide a separate input control.

Run this notebook from `2d-tracking` or its `notebooks` directory. It reads
saved TrackEval results and tracker timings; it does not rerun the models.


In [1]:
from __future__ import annotations

import csv
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    display
except NameError:
    display = print


def find_tracking_root():
    override = os.environ.get("AGHRI_TRACK_ROOT")
    if override:
        return Path(override).resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "reports/aghri_test/trackeval_summary_by_source.csv").exists():
            return candidate
    raise FileNotFoundError("Run from the 2d-tracking repository or set AGHRI_TRACK_ROOT")


ROOT = find_tracking_root()
DATA = ROOT / "reports/aghri_test"
OUT = Path(os.environ.get("AGHRI_TRACK_PAPER_OUT", DATA / "paper_results"))
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

TRACKERS = [
    ("boxmot_bytetrack", "ByteTrack"),
    ("boxmot_ocsort", "OC-SORT"),
    ("boxmot_deepocsort", "Deep OC-SORT"),
    ("boxmot_botsort", "BoT-SORT"),
    ("boxmot_boosttrack", "BoostTrack"),
    ("boxmot_strongsort", "StrongSORT"),
    ("deepsort", "DeepSORT"),
    ("norfair", "Norfair"),
]
TRACKER_NAMES = dict(TRACKERS)
TRACKER_ORDER = [key for key, _ in TRACKERS]
DETECTOR_SOURCES = ("zed_yolov8s_pretrained", "fisheye_yolov8s_pretrained")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 10,
    "figure.facecolor": "white", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save_figure(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


def save_csv(frame, name):
    frame.to_csv(OUT / name, index=False, float_format="%.10f")

## Load the saved test results

Rate metrics in the paper tables are equal means over the 12 camera-specific
sequences for each tracker. The input-setting comparison averages the 96
tracker-by-camera-specific-sequence results in each setting. Identity switches
are summed. Tracker-processing throughput is the equal mean of the 12 per-run
frame rates. TrackEval MOTP is matched-box IoU, where higher is better.

For the saved HOTA values and the distinction between equal means and pooled
metrics, see the [report guide](../reports/aghri_test/README.md).


In [2]:
records = []
for source in (*DETECTOR_SOURCES, "gt"):
    for tracker in TRACKER_ORDER:
        path = DATA / "trackeval_detailed" / source / f"{tracker}.csv"
        with path.open(newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                if row["seq"] == "COMBINED":
                    continue
                stream = row["seq"]
                if stream.endswith("_zed_rgb"):
                    sequence = stream[:-len("_zed_rgb")]
                    camera = "ZED RGB"
                elif stream.endswith("_front_fisheye"):
                    sequence = stream[:-len("_front_fisheye")]
                    camera = "Front fisheye"
                else:
                    raise ValueError(stream)
                records.append({
                    "input": "Ground-truth-based" if source == "gt" else "Detector-based",
                    "source": source, "tracker": tracker,
                    "sequence": sequence, "camera": camera,
                    "mota": float(row["MOTA"]),
                    "motp": float(row["MOTP"]),
                    "idf1": float(row["IDF1"]),
                    "idp": float(row["IDP"]),
                    "idr": float(row["IDR"]),
                    "precision": float(row["CLR_Pr"]),
                    "recall": float(row["CLR_Re"]),
                    "idsw": int(float(row["IDSW"])),
                    "hota": float(row["HOTA___AUC"]),
                    "evaluated_frames": int(float(row["CLR_Frames"])),
                })
results = pd.DataFrame(records)
assert len(results) == 192
assert results.groupby(["input", "tracker"]).size().eq(12).all()

timing = pd.read_csv(DATA / "tracker_throughput_per_run.csv")
timing = timing[timing["source"].isin(DETECTOR_SOURCES)].copy()
assert len(timing) == 96
assert timing.groupby("tracker").size().eq(12).all()

## Overall detector-based tracking table

Each row represents one tracker. MOTA, MOTP, identity metrics, and throughput
are averaged equally across its 12 camera-specific sequences; IDSW is summed.
The table uses the YOLOv8s detector inputs selected for the paper.


In [3]:
detector = results[results["input"] == "Detector-based"].copy()
control = results[results["input"] == "Ground-truth-based"].copy()
overall = detector.groupby("tracker", sort=False).agg(
    mota=("mota", "mean"), motp=("motp", "mean"),
    idf1=("idf1", "mean"), idp=("idp", "mean"),
    idr=("idr", "mean"), idsw=("idsw", "sum"),
    hota=("hota", "mean"), camera_streams=("sequence", "size"),
)
overall["tracker_fps"] = timing.groupby("tracker")["tracking_fps"].mean()
overall = overall.reindex(TRACKER_ORDER).reset_index()
overall.insert(1, "tracker_name", overall["tracker"].map(TRACKER_NAMES))
assert overall["camera_streams"].eq(12).all()
save_csv(overall, "overall_detector_based.csv")
display(overall[["tracker_name", "mota", "motp", "idf1", "idp", "idr", "idsw", "tracker_fps"]])

,tracker_name,mota,motp,idf1,idp,idr,idsw,tracker_fps
0,ByteTrack,0.902016,0.844424,0.782573,0.785797,0.779502,60,2056.286949
1,OC-SORT,0.920087,0.889203,0.724503,0.728997,0.720305,73,1879.390823
2,Deep OC-SORT,0.920578,0.889555,0.722739,0.729468,0.716371,75,33.843988
3,BoT-SORT,0.935195,0.881991,0.765053,0.769157,0.761093,47,39.162824
4,BoostTrack,0.918056,0.878236,0.755618,0.766146,0.746095,46,32.432660
5,StrongSORT,0.897023,0.886915,0.756050,0.752929,0.760272,85,18.304460
6,DeepSORT,0.895666,0.845636,0.747710,0.753225,0.742483,60,104.790468
7,Norfair,0.837369,0.866993,0.727883,0.726026,0.730353,46,2660.874551


## Detector-based versus ground-truth-based input table

Each setting has 96 tracker-by-camera-specific-sequence results. The rate
metrics are equal-run means, and IDSW is summed. The ground-truth-box control
changes the tracker inputs, not the evaluation ground truth.


In [4]:
comparison = results.groupby("input", sort=False).agg(
    mota=("mota", "mean"), idf1=("idf1", "mean"),
    precision=("precision", "mean"), recall=("recall", "mean"),
    idsw=("idsw", "sum"), tracker_camera_runs=("sequence", "size"),
)
comparison = comparison.reindex(["Detector-based", "Ground-truth-based"])
assert comparison["tracker_camera_runs"].eq(96).all()
delta = comparison.loc["Ground-truth-based"] - comparison.loc["Detector-based"]
comparison.loc["Difference (GT - detector)"] = delta
comparison = comparison.reset_index()
save_csv(comparison, "detector_vs_gt_overall.csv")
display(comparison)

,input,mota,idf1,precision,recall,idsw,tracker_camera_runs
0,Detector-based,0.903249,0.747766,0.960906,0.947185,492.0,96.0
1,Ground-truth-based,0.959420,0.745463,0.989388,0.974819,509.0,96.0
2,Difference (GT - detector),0.056171,-0.002304,0.028483,0.027634,17.0,0.0


## Figure 1. Key tracker metrics

This figure shows detector-based MOTA, IDF1, and the total number of identity
switches for all eight trackers. The plotted values come from the overall
detector-based table above.


In [5]:
bar_colors = ["#4B8FB7", "#306883", "#829CA5", "#B84E3D",
              "#CF8245", "#9B6D60", "#A68E77", "#667175"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.7))
x = np.arange(len(overall))
for ax, metric, title, direction, fmt in zip(
    axes, ("mota", "idf1", "idsw"),
    ("(a) MOTA", "(b) IDF1", "(c) Total IDSW"),
    ("higher is better", "higher is better", "lower is better"),
    (".4f", ".4f", ".0f"),
):
    values = overall[metric].to_numpy()
    bars = ax.bar(x, values, color=bar_colors, width=0.72)
    best_value = values.min() if metric == "idsw" else values.max()
    best_indices = set(np.flatnonzero(np.isclose(values, best_value)).tolist())
    for i in best_indices:
        bars[i].set_edgecolor("#222222")
        bars[i].set_linewidth(1.7)
    ax.set_title(title, pad=14)
    ax.set_ylabel(metric.upper() if metric == "idsw" else metric.upper())
    ax.set_xticks(x, overall["tracker_name"], rotation=42, ha="right")
    ax.set_ylim(0, max(1.05, values.max() * 1.17) if metric != "idsw" else values.max() * 1.18)
    ax.yaxis.grid(True, alpha=0.22)
    ax.set_axisbelow(True)
    ax.text(0.99, 0.98, direction, transform=ax.transAxes,
            ha="right", va="top", fontsize=8, color="#777777")
    for i, (bar, value) in enumerate(zip(bars, values)):
        ax.annotate(format(value, fmt),
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center",
                    va="bottom", fontsize=8, fontweight="bold" if i in best_indices else "normal")
fig.tight_layout(w_pad=2.2)
save_figure(fig, "aghri_2d_tracking_key_metrics_new")
from shutil import copy2
notebook_figures = ROOT / "notebooks" / "figures"
notebook_figures.mkdir(parents=True, exist_ok=True)
for extension in ("png", "pdf"):
    name = f"aghri_2d_tracking_key_metrics_new.{extension}"
    copy2(FIGURES / name, notebook_figures / name)

## Figure 2. Camera-specific tracking across all six test sequences

Each bar is the equal mean of 48 tracker-by-sequence results (six sequences and
eight trackers) for one camera and input setting. These results compare the
complete camera-specific pipelines; the detector-based results include the
separately selected checkpoint for each camera.


In [6]:
from shutil import copy2

camera_summary = (
    results.groupby(["input", "camera"], sort=False)
    .agg(mota=("mota", "mean"), idf1=("idf1", "mean"), runs=("tracker", "size"))
    .reset_index()
)
assert len(camera_summary) == 4
assert camera_summary["runs"].eq(48).all()

settings = ["Detector-based", "Ground-truth-based"]
cameras = ["Front fisheye", "ZED RGB"]
colors = {"Front fisheye": "#596BC2", "ZED RGB": "#27A69A"}
x_camera = np.arange(len(settings))
width = 0.32
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.4), sharey=True)
for ax, metric, title in zip(axes, ("mota", "idf1"), ("(a) MOTA", "(b) IDF1")):
    for j, camera in enumerate(cameras):
        values = [camera_summary.loc[
            (camera_summary["input"] == setting) & (camera_summary["camera"] == camera),
            metric,
        ].iloc[0] for setting in settings]
        bars = ax.bar(x_camera + (j - 0.5) * width, values, width=width,
                      color=colors[camera], label=camera, zorder=3)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + 0.012,
                    f"{value:.4f}", ha="center", va="bottom", fontsize=10)
    ax.set_title(title, fontsize=14, pad=12)
    ax.set_ylabel(metric.upper(), fontsize=11)
    ax.set_xticks(x_camera, settings)
    ax.set_ylim(0, 1.075)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.yaxis.grid(True, alpha=0.22)
    ax.set_axisbelow(True)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=True,
           bbox_to_anchor=(0.5, 0.985))
fig.subplots_adjust(left=0.07, right=0.99, bottom=0.15, top=0.81, wspace=0.16)
name = "aghri_2d_tracking_camera_input_comparison_new"
save_figure(fig, name)
notebook_figures = ROOT / "notebooks" / "figures"
notebook_figures.mkdir(parents=True, exist_ok=True)
for extension in ("png", "pdf"):
    copy2(FIGURES / f"{name}.{extension}", notebook_figures / f"{name}.{extension}")
display(camera_summary)


,input,camera,mota,idf1,runs
0,Detector-based,ZED RGB,0.872256,0.746629,48
1,Detector-based,Front fisheye,0.934242,0.748903,48
2,Ground-truth-based,Front fisheye,0.978034,0.750386,48
3,Ground-truth-based,ZED RGB,0.940805,0.740540,48


## Generated paper artifacts

The notebook saves two [result CSVs](../reports/aghri_test/paper_results/)
under `reports/aghri_test/paper_results/`. It saves both figures as PNG and
PDF in that report's `figures` directory. Copies of the figures are also kept
beside the notebook in `notebooks/figures/`.

### Overall tracker key metrics

![2D tracking key metrics](figures/aghri_2d_tracking_key_metrics_new.png)

### Camera-specific tracking comparison

![Camera-specific tracking comparison](figures/aghri_2d_tracking_camera_input_comparison_new.png)
